# LDA K=6 Stability Analysis (Comprehensive & Cached)

This notebook evaluates the stability of the final $K=6$ LDA persona model across 5 independent initializations (seeds: `[42, 123, 456, 789, 999]`).

To avoid repeating the time-consuming model fitting, this notebook:
1. **Caches Models and Assignments**: Pre-fitted models and document assignments are saved to `ablations/saved_models/` and loaded automatically if they exist.
2. **Matched vs. Mismatched Baselines**: We compare matched persona similarity (Hungarian aligned) against the baseline similarity of mismatched (different) personas.
3. **Per-Persona Breakdown**: We report the stability statistics for each individual persona aligned to a reference run (Seed 42).
4. **Keyword Overlap (Top-5, Top-10, Top-15, Top-20)**: We report the overlap of keywords between matched and mismatched topic pairs at different thresholds.


In [1]:
import numpy as np
import pandas as pd
import ast
import os
import pickle
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from scipy.optimize import linear_sum_assignment
from scipy.stats import entropy
from sklearn.metrics.pairwise import cosine_similarity
import itertools

# Safe JS Distance function to prevent numerical precision NaNs
def safe_js_distance(p, q):
    p = np.clip(p, 1e-10, 1.0)
    q = np.clip(q, 1e-10, 1.0)
    
    sum_p = np.sum(p)
    sum_q = np.sum(q)
    
    p = p / (sum_p if sum_p > 0 else 1.0)
    q = q / (sum_q if sum_q > 0 else 1.0)
    
    m = 0.5 * (p + q)
    
    kl_pm = entropy(p, m, base=2)
    kl_qm = entropy(q, m, base=2)
    
    if np.isnan(kl_pm) or np.isinf(kl_pm):
        kl_pm = 0.0
    if np.isnan(kl_qm) or np.isinf(kl_qm):
        kl_qm = 0.0
        
    js_div = 0.5 * kl_pm + 0.5 * kl_qm
    js_div = np.clip(js_div, 0.0, 1.0)
    
    return np.sqrt(js_div)

def safe_js_similarity(p, q):
    return 1.0 - safe_js_distance(p, q)

print("Loading data...")
df_post = pd.read_csv('../data/processed/posts_w_labels.csv')
df_post['Label'] = df_post['Label'].apply(ast.literal_eval)
df_post = df_post[df_post.Label.apply(len) > 0]

post2labels = df_post['Label'].to_dict()

documents = []
post_ids = []
for post_id, topics in post2labels.items():
    document = ' '.join([topic.replace(' ', '_') for topic in topics])
    documents.append(document)
    post_ids.append(post_id)
    
print(f"Loaded {len(documents)} documents for stability analysis.")

# Create document-term matrix
vectorizer = CountVectorizer(
    token_pattern=r'[^_\s]+(?:_[^_\s]+)*',  # Match topics with underscores
    lowercase=False,
    min_df=2,
    max_df=0.95
)
doc_term_matrix = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()
print(f"Doc-term matrix shape: {doc_term_matrix.shape}, Vocabulary size: {len(feature_names)}")


Loading data...
Loaded 10852 documents for stability analysis.
Doc-term matrix shape: (10852, 48), Vocabulary size: 48


In [2]:
seeds = [42, 123, 456, 789, 999]
models = {}
assignments = {}
topic_words = {}
K = 6
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

print("Loading or fitting LDA models for 5 different seeds...")
for seed in seeds:
    model_path = os.path.join(save_dir, f"lda_model_{seed}.pkl")
    assign_path = os.path.join(save_dir, f"lda_assign_{seed}.npy")
    
    if os.path.exists(model_path) and os.path.exists(assign_path):
        print(f"Loading cached LDA model and assignments for seed={seed}...")
        with open(model_path, 'rb') as f:
            lda = pickle.load(f)
        doc_topic_probs = np.load(assign_path)
    else:
        print(f"Fitting LDA with seed={seed}...")
        lda = LatentDirichletAllocation(
            n_components=K,
            random_state=seed,
            max_iter=100,
            learning_method='batch',
            evaluate_every=10,
            verbose=0
        )
        doc_topic_probs = lda.fit_transform(doc_term_matrix)
        
        # Cache to disk
        with open(model_path, 'wb') as f:
            pickle.dump(lda, f)
        np.save(assign_path, doc_topic_probs)
        
    models[seed] = lda
    assignments[seed] = doc_topic_probs
    
    # Get top 20 keywords for each of the 6 topics (allows top-5, top-10, top-15, and top-20 evaluation)
    run_topics = []
    for topic_idx in range(K):
        topic_probs = lda.components_[topic_idx]
        top_indices = topic_probs.argsort()[-20:][::-1]
        top_keywords = [feature_names[i].replace('_', ' ') for i in top_indices]
        run_topics.append(top_keywords)
    topic_words[seed] = run_topics
print("All models loaded/fitted successfully!")


Loading or fitting LDA models for 5 different seeds...
Loading cached LDA model and assignments for seed=42...
Loading cached LDA model and assignments for seed=123...
Loading cached LDA model and assignments for seed=456...
Loading cached LDA model and assignments for seed=789...
Loading cached LDA model and assignments for seed=999...
All models loaded/fitted successfully!


In [3]:
print("--- Topic Profile Stability Analysis (Beta) ---")
pairs = list(itertools.combinations(seeds, 2))

matched_cos_all = []
mismatched_cos_all = []
matched_js_all = []
mismatched_js_all = []

# Track per-persona stability aligned to Seed 42
ref_seed = 42
per_persona_cos = {i: [] for i in range(K)}
per_persona_js = {i: [] for i in range(K)}

beta_ref = models[ref_seed].components_ / models[ref_seed].components_.sum(axis=1, keepdims=True)

# Loop over pairs
for seed1, seed2 in pairs:
    lda1 = models[seed1]
    lda2 = models[seed2]
    
    beta1 = lda1.components_ / lda1.components_.sum(axis=1, keepdims=True)
    beta2 = lda2.components_ / lda2.components_.sum(axis=1, keepdims=True)
    
    # Cosine Similarity Matrix
    cos_matrix = cosine_similarity(lda1.components_, lda2.components_)
    row_ind, col_ind = linear_sum_assignment(-cos_matrix)
    
    # Matched vs Mismatched
    for r in range(K):
        matched_c = col_ind[r]
        matched_cos_all.append(cos_matrix[r, matched_c])
        for c in range(K):
            if c != matched_c:
                mismatched_cos_all.append(cos_matrix[r, c])
                
    # JS Similarity Matrix
    js_sim_matrix = np.zeros((K, K))
    for r in range(K):
        for c in range(K):
            js_sim_matrix[r, c] = safe_js_similarity(beta1[r], beta2[c])
            
    row_ind_js, col_ind_js = linear_sum_assignment(-js_sim_matrix)
    for r in range(K):
        matched_c = col_ind_js[r]
        matched_js_all.append(js_sim_matrix[r, matched_c])
        for c in range(K):
            if c != matched_c:
                mismatched_js_all.append(js_sim_matrix[r, c])

# Run specific comparisons to reference Seed 42 for per-persona breakdown
for seed in seeds:
    if seed == ref_seed:
        continue
    beta_seed = models[seed].components_ / models[seed].components_.sum(axis=1, keepdims=True)
    
    # Cosine match to reference
    cos_matrix = cosine_similarity(models[ref_seed].components_, models[seed].components_)
    row_ind, col_ind = linear_sum_assignment(-cos_matrix)
    for r, c in zip(row_ind, col_ind):
        per_persona_cos[r].append(cos_matrix[r, c])
        
    # JS match to reference
    js_sim_matrix = np.zeros((K, K))
    for r in range(K):
        for c in range(K):
            js_sim_matrix[r, c] = safe_js_similarity(beta_ref[r], beta_seed[c])
    row_ind_js, col_ind_js = linear_sum_assignment(-js_sim_matrix)
    for r, c in zip(row_ind_js, col_ind_js):
        per_persona_js[r].append(js_sim_matrix[r, c])

print(f"Average Matched Cosine Sim (Beta):   {np.mean(matched_cos_all):.4f} (std: {np.std(matched_cos_all):.4f})")
print(f"Average Mismatched Cosine Sim (Beta): {np.mean(mismatched_cos_all):.4f} (std: {np.std(mismatched_cos_all):.4f})")
print(f"Average Matched JS Sim (Beta):       {np.mean(matched_js_all):.4f} (std: {np.std(matched_js_all):.4f})")
print(f"Average Mismatched JS Sim (Beta):     {np.mean(mismatched_js_all):.4f} (std: {np.std(mismatched_js_all):.4f})")

print("\n--- Per-Persona Stability Breakdown (Aligned to Seed 42) ---")
for r in range(K):
    print(f"Persona {r+1}:")
    print(f"  Cosine Sim: {np.mean(per_persona_cos[r]):.4f} (std: {np.std(per_persona_cos[r]):.4f})")
    print(f"  JS Sim:     {np.mean(per_persona_js[r]):.4f} (std: {np.std(per_persona_js[r]):.4f})")


--- Topic Profile Stability Analysis (Beta) ---
Average Matched Cosine Sim (Beta):   0.6392 (std: 0.2191)
Average Mismatched Cosine Sim (Beta): 0.1823 (std: 0.1568)
Average Matched JS Sim (Beta):       0.3989 (std: 0.1569)
Average Mismatched JS Sim (Beta):     0.1243 (std: 0.0873)

--- Per-Persona Stability Breakdown (Aligned to Seed 42) ---
Persona 1:
  Cosine Sim: 0.7122 (std: 0.1674)
  JS Sim:     0.4384 (std: 0.1099)
Persona 2:
  Cosine Sim: 0.6124 (std: 0.1637)
  JS Sim:     0.3361 (std: 0.1456)
Persona 3:
  Cosine Sim: 0.4092 (std: 0.0669)
  JS Sim:     0.2774 (std: 0.0249)
Persona 4:
  Cosine Sim: 0.5912 (std: 0.1808)
  JS Sim:     0.3832 (std: 0.1125)
Persona 5:
  Cosine Sim: 0.6357 (std: 0.1371)
  JS Sim:     0.4030 (std: 0.1116)
Persona 6:
  Cosine Sim: 0.9564 (std: 0.0340)
  JS Sim:     0.6578 (std: 0.0756)


In [4]:
print("--- Topic Keyword Overlap Analysis (Top 5, Top 10, Top 15, Top 20) ---")

for N_words in [5, 10, 15, 20]:
    print(f"\nEvaluating top-{N_words} keyword overlap:")
    matched_overlap_all = []
    mismatched_overlap_all = []

    for seed1, seed2 in pairs:
        lda1 = models[seed1]
        lda2 = models[seed2]
        
        beta1 = lda1.components_ / lda1.components_.sum(axis=1, keepdims=True)
        beta2 = lda2.components_ / lda2.components_.sum(axis=1, keepdims=True)
        
        # Use JSD to align topics for consistency
        js_matrix = np.zeros((K, K))
        for r in range(K):
            for c in range(K):
                js_matrix[r, c] = safe_js_distance(beta1[r], beta2[c])
        row_ind, col_ind = linear_sum_assignment(js_matrix)
        
        # Matched overlaps
        for r in range(K):
            matched_c = col_ind[r]
            w1 = topic_words[seed1][r][:N_words]
            w2 = topic_words[seed2][matched_c][:N_words]
            overlap = len(set(w1).intersection(set(w2))) / float(N_words)
            matched_overlap_all.append(overlap)
            
            # Mismatched overlaps
            for c in range(K):
                if c != matched_c:
                    w2_mis = topic_words[seed2][c][:N_words]
                    overlap_mis = len(set(w1).intersection(set(w2_mis))) / float(N_words)
                    mismatched_overlap_all.append(overlap_mis)

    print(f"  Average Matched Overlap:   {100*np.mean(matched_overlap_all):.1f}% (std: {100*np.std(matched_overlap_all):.1f}%)")
    print(f"  Average Mismatched Overlap: {100*np.mean(mismatched_overlap_all):.1f}% (std: {100*np.std(mismatched_overlap_all):.1f}%)")


--- Topic Keyword Overlap Analysis (Top 5, Top 10, Top 15, Top 20) ---

Evaluating top-5 keyword overlap:
  Average Matched Overlap:   50.3% (std: 20.8%)
  Average Mismatched Overlap: 20.7% (std: 15.6%)

Evaluating top-10 keyword overlap:
  Average Matched Overlap:   50.2% (std: 14.7%)
  Average Mismatched Overlap: 25.2% (std: 13.0%)

Evaluating top-15 keyword overlap:
  Average Matched Overlap:   54.4% (std: 11.9%)
  Average Mismatched Overlap: 33.7% (std: 12.8%)

Evaluating top-20 keyword overlap:
  Average Matched Overlap:   61.8% (std: 7.9%)
  Average Mismatched Overlap: 47.1% (std: 11.0%)


In [5]:
print("--- Document-Topic Allocation Stability Analysis (Theta) ---")

matched_doc_cos_all = []
mismatched_doc_cos_all = []
matched_doc_js_all = []
mismatched_doc_js_all = []

for seed1, seed2 in pairs:
    probs1 = assignments[seed1]
    probs2 = assignments[seed2]
    
    lda1 = models[seed1]
    lda2 = models[seed2]
    
    beta1 = lda1.components_ / lda1.components_.sum(axis=1, keepdims=True)
    beta2 = lda2.components_ / lda2.components_.sum(axis=1, keepdims=True)
    
    # Align topics using JSD
    js_matrix = np.zeros((K, K))
    for r in range(K):
        for c in range(K):
            js_matrix[r, c] = safe_js_distance(beta1[r], beta2[c])
    row_ind, col_ind = linear_sum_assignment(js_matrix)
    
    probs2_aligned = probs2[:, col_ind]
    
    # Matched Cosine similarities
    for r in range(K):
        matched_c = col_ind[r]
        vec1 = probs1[:, r]
        vec2 = probs2[:, matched_c]
        cos_val = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2) + 1e-10)
        matched_doc_cos_all.append(cos_val)
        
        # Mismatched Cosine
        for c in range(K):
            if c != matched_c:
                vec2_mis = probs2[:, c]
                cos_val_mis = np.dot(vec1, vec2_mis) / (np.linalg.norm(vec1) * np.linalg.norm(vec2_mis) + 1e-10)
                mismatched_doc_cos_all.append(cos_val_mis)
                
    # JS soft assignments over documents
    dists_match = [safe_js_distance(probs1[i], probs2_aligned[i]) for i in range(len(probs1))]
    matched_doc_js_all.append(1.0 - np.mean(dists_match))
    
    # Mismatched JS soft assignments (all shifts)
    pair_mis_js = []
    for shift in range(1, K):
        probs2_shifted = np.roll(probs2_aligned, shift=shift, axis=1)
        dists_mis = [safe_js_distance(probs1[i], probs2_shifted[i]) for i in range(len(probs1))]
        pair_mis_js.append(1.0 - np.mean(dists_mis))
    mismatched_doc_js_all.append(np.mean(pair_mis_js))

print(f"Average Matched Cosine Sim (Theta):   {np.mean(matched_doc_cos_all):.4f} (std: {np.std(matched_doc_cos_all):.4f})")
print(f"Average Mismatched Cosine Sim (Theta): {np.mean(mismatched_doc_cos_all):.4f} (std: {np.std(mismatched_doc_cos_all):.4f})")
print(f"Average Matched JS Sim (Theta):       {np.mean(matched_doc_js_all):.4f} (std: {np.std(matched_doc_js_all):.4f})")
print(f"Average Mismatched JS Sim (Theta):     {np.mean(mismatched_doc_js_all):.4f} (std: {np.std(mismatched_doc_js_all):.4f})")


--- Document-Topic Allocation Stability Analysis (Theta) ---
Average Matched Cosine Sim (Theta):   0.6672 (std: 0.1389)
Average Mismatched Cosine Sim (Theta): 0.3775 (std: 0.0858)
Average Matched JS Sim (Theta):       0.6539 (std: 0.0483)
Average Mismatched JS Sim (Theta):     0.4255 (std: 0.0083)
